# 03 — The Evoformer (`evoformer.py`)
The trunk: **m and z exchange information** for `n_evo` blocks. Each block (Algorithm 6):
1. MSA stack: row attention (biased by z!) → column attention → transition
2. m → z: outer product mean
3. pair stack: triangle multiplication ×2 → triangle attention ×2 → transition

The **extra MSA** runs first in its own cheaper tower (no column attention, c_e=32) and only refines z.

## Stage map

```text
Extra MSA e --> cheaper Evoformer blocks --------------------+
                                                              v
MSA m --> row attention (+ z bias) --> column attention --> transition
  |                                                               |
  +-------------------- outer-product mean ------------------------+--> pair z
                                                                      |
                         triangle multiplication: outgoing + incoming |
                                                                      v
                         triangle attention: start + end --> transition
                                                                      |
                           repeat main block --------------------------+

Output: refined m and z
```

**Read alongside:** `../src/af2_from_scratch/evoformer.py`. The central idea is two-way communication: `z` biases MSA attention, while the outer-product mean sends MSA information back into `z`.

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")  # package source lives one level up
torch.manual_seed(0)
plt.rcParams["figure.figsize"] = (8, 4)

In [ ]:
from af2_from_scratch.feature_extraction import msa_features, sample_batch
from af2_from_scratch.feature_embedding import InputEmbedder
from af2_from_scratch.evoformer import (
    Evoformer,
    OuterProductMean,
    TriMult,
)
from af2_from_scratch import AF2Config

cfg = AF2Config()
f = msa_features("../examples/tautomerase/alignment.a3m")
b = sample_batch(f, cfg.n_clu, cfg.n_ext, seed=0)
m, z, e = InputEmbedder(cfg)(b)
print("m:", tuple(m.shape), " z:", tuple(z.shape), " e:", tuple(e.shape))

## 1. Outer product mean — the bridge m → z
`einsum('sic,sjd->ijcd', a, b) / S`: every residue pair (i,j) gets the product of its two MSA columns, averaged over sequences. This is how co-evolution enters the pair representation: if positions i and j *co-vary across sequences*, z_ij notices.

In [ ]:
opm = OuterProductMean(cfg.c_m, cfg.c_z)
dz = opm(m)
print("OPM output:", tuple(dz.shape))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(z[..., 0].detach(), cmap="RdBu")
ax[0].set_title("z before (channel 0)")
ax[1].imshow((z + dz)[..., 0].detach(), cmap="RdBu")
ax[1].set_title("z after OPM")
plt.show()

## 2. Triangle updates — geometry as a consistency rule
If residue i is near k, and k is near j, then i is probably near j. Triangle multiplication updates z_ij by mixing over all k: `einsum('ikc,jkc->ijc')` (outgoing) or `'kic,kjc->ijc'` (incoming). Gates decide what passes.

In [ ]:
tri = TriMult(cfg.c_z, outgoing=True)
dz2 = tri(z)
print(
    "triangle update:",
    tuple(dz2.shape),
    " — one einsum replaces a triple loop over (i,j,k)",
)

## 3. Run the full trunk
Extra MSA tower first (refines z), then 4 main blocks. Watch the parameter count — this is 45× smaller than AF2's trunk with every op type intact.

In [ ]:
evo = Evoformer(cfg)
m_out, z_out = evo(m, z, e)
print("after evoformer: m", tuple(m_out.shape), " z", tuple(z_out.shape))
print(
    f"evoformer params: {sum(p.numel() for p in evo.parameters()) / 1e6:.2f}M  (AF2: ~80M)"
)
print("representations actually changed:", not torch.allclose(z, z_out))

## 4. Which ops cost what?
A quick breakdown — triangle ops dominate despite the tiny size, exactly like in AF2.

In [ ]:
blk = evo.blocks[0]
for name, mod in [
    ("row attn", blk.row),
    ("col attn", blk.col),
    ("transition m", blk.tm),
    ("OPM", blk.opm),
    ("tri mult x2", torch.nn.ModuleList([blk.tmo, blk.tmi])),
    ("tri attn x2", torch.nn.ModuleList([blk.tas, blk.tae])),
    ("transition z", blk.tp),
]:
    print(f"{name:14s} {sum(p.numel() for p in mod.parameters()) / 1e3:7.1f}k")

**Next:** `04_geometry.ipynb` — the coordinate tools needed to understand how learned representations become a 3D fold.